In [1]:
!pip install -q -U transformers accelerate requests

import os
import json
import glob
import time
import requests
import numpy as np
import pandas as pd
import torch
from PIL import Image
from scipy.stats import spearmanr
from transformers import AutoProcessor, AutoModelForImageTextToText

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
device: cuda


In [2]:
MODEL_NAME = "OpenGVLab/InternVL3-8B-hf"

processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    max_memory={0: "7GiB", 1: "13GiB"},
)
model.eval()
for p_ in model.parameters():
    p_.requires_grad_(False)

ln_f = model.model.language_model.norm
lm_head = model.lm_head
layers = model.model.language_model.layers

n_layer = model.config.text_config.num_hidden_layers
n_head = model.config.text_config.num_attention_heads
hidden_dim = model.config.text_config.hidden_size
head_dim = layers[0].self_attn.o_proj.in_features // n_head
IMAGE_TOKEN_ID = model.config.image_token_id
IMAGE_SEQ = processor.image_seq_length
assert len(layers) == n_layer
print(n_layer, "language model layers,", n_head, "heads,", head_dim, "head dim,", model.config.text_config.num_key_value_heads, "kv heads, image token id", IMAGE_TOKEN_ID, "| tokens per tile", IMAGE_SEQ)

processor_config.json:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/481 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.96k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/6.86k [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/781 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie model.language_model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/126 [00:00<?, ?B/s]

28 language model layers, 28 heads, 128 head dim, 4 kv heads, image token id 151667 | tokens per tile 256


In [3]:
INPUT = "/kaggle/input/datasets/nocturnalnerd18"
OWN_INPUT = "/kaggle/input/datasets/ponishbhatia3"
AMBER_IMAGE_DIR = "/kaggle/input/datasets/nocturnalnerd18/amber-hallucination/image"
VG_ROOT = "/kaggle/input/datasets/sajjad2goudarzi/the-visual-genome-dataset"
VG_INDEX = {}
for dp, _, fns in os.walk(VG_ROOT):
    for fn in fns:
        if fn.lower().endswith(".jpg"):
            VG_INDEX[fn[:-4]] = os.path.join(dp, fn)
print(len(VG_INDEX), "Visual Genome images indexed")
assert len(VG_INDEX) > 100000, "check VG_ROOT and that the visual genome dataset is attached"

NOISE_CANDIDATES = [3.0, 4.0, 5.0, 6.0]
FINALISTS = [3.0, 4.0, 6.0]
LEAN_MAX = 2.0
MAX_T_CAL = 800
MAX_T_ACC = 1800
TAIL_N = 5
GRAD_SCALE_CANDIDATES = [32.0, 8.0, 1.0]
GRAD_SCALE = GRAD_SCALE_CANDIDATES[0]
RECALL_K = 20
HEADROOM_MAX = 20000.0
GROUPS = ["visual", "text", "tail"]
N_ACC_PER_COMBO = 6
N_HEAD_TOP = 20
N_HEAD_RAND = 20
N_SEED_Q = 3
MIN_TE = 1.0
COMBOS = [("attribute", "amber"), ("relation", "amber"), ("relation", "reefknot")]


def load_image(path):
    return Image.open(path).convert("RGB")


def build_inputs(image, text):
    conversation = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": text}]}]
    prompt_text = processor.apply_chat_template(conversation, add_generation_prompt=True)
    inputs = processor(text=prompt_text, images=image, return_tensors="pt").to(model.device, torch.float16)
    inputs.pop("token_type_ids", None)
    return inputs


def find_vg_image_path(image_id):
    return VG_INDEX.get(str(image_id))


def resolve_image_path(q):
    qid = q["question_id"]
    if qid.startswith("amber_"):
        return os.path.join(AMBER_IMAGE_DIR, f"{q['image_id']}.jpg")
    if qid.startswith("reefknot_"):
        return find_vg_image_path(q["image_id"])
    return None


with open(f"{INPUT}/vlm-questions/all_questions.json") as f:
    all_questions = json.load(f)
assert len(all_questions) == 17492, "attach vlm-questions version 2"
question_by_id = {q["question_id"]: q for q in all_questions}

with open(f"{INPUT}/dla-intern/dla_meta.json") as f:
    dla_meta = json.load(f)
YES_IDS, NO_IDS = dla_meta["yes_ids"], dla_meta["no_ids"]
for prefix in YES_IDS:
    yes_words = [processor.tokenizer.decode([i]).strip() for i in YES_IDS[prefix]]
    no_words = [processor.tokenizer.decode([i]).strip() for i in NO_IDS[prefix]]
    print(prefix, YES_IDS[prefix], yes_words, NO_IDS[prefix], no_words)
    assert {w.lower() for w in yes_words} == {"yes"} and {w.lower() for w in no_words} == {"no"}, (yes_words, no_words)
assert set(YES_IDS) == {"amber_", "reefknot_"}, set(YES_IDS)

with open(f"{OWN_INPUT}/atp-intern-selection/selection.json") as f:
    selection = json.load(f)
all_sel = [qid for t in selection["selected"] for qid in selection["selected"][t]]
print(len(all_sel), "selected questions")
need = {question_by_id[q]["image_id"] for q in all_sel if q.startswith("reefknot_")}
n_miss = sum(str(i) not in VG_INDEX for i in need)
print(len(need), "reefknot images needed,", n_miss, "missing")
assert n_miss == 0, "some reefknot images are not in the visual genome dataset"

rng = np.random.default_rng(0)
_prep_cache = {}


def prep(qid):
    if qid in _prep_cache:
        return _prep_cache[qid]
    q = question_by_id[qid]
    inputs = build_inputs(load_image(resolve_image_path(q)), q["question_text"])
    ids = inputs["input_ids"]
    T = ids.shape[1]
    vis = ids[0] == IMAGE_TOKEN_ID
    tail = torch.zeros(T, dtype=torch.bool, device=ids.device)
    tail[-TAIL_N:] = True
    text = ~vis & ~tail
    assert int(vis.sum()) == inputs["pixel_values"].shape[0] * IMAGE_SEQ and not bool((vis & tail).any())
    prefix = qid.split("_")[0] + "_"
    _prep_cache[qid] = {
        "qid": qid, "T": T, "inputs": inputs, "vis": vis, "yes_ids": YES_IDS[prefix], "no_ids": NO_IDS[prefix],
        "masks": {"visual": vis.float().cpu(), "text": text.float().cpu(), "tail": tail.float().cpu()},
        "sign": 1.0 if q["ground_truth_answer"].strip().lower() == "yes" else -1.0,
    }
    return _prep_cache[qid]


def pick(stratum, n, combos, max_T=None):
    pool = [
        q for q in all_sel
        if selection["stratum"][q] == stratum
        and (question_by_id[q]["hallucination_type"], selection["source"][q]) in combos
    ]
    out = []
    for q in [str(x) for x in rng.permutation(pool)]:
        path = resolve_image_path(question_by_id[q])
        if path is None or not os.path.exists(path):
            continue
        if max_T is not None and prep(q)["T"] > max_T:
            continue
        out.append(q)
        if len(out) == n:
            break
    return out


acc_set = []
for combo in COMBOS:
    acc_set += pick("true_yes", N_ACC_PER_COMBO, [combo], MAX_T_ACC) + pick("true_no", N_ACC_PER_COMBO, [combo], MAX_T_ACC)
cal_set = (
    pick("false_yes", 3, [("attribute", "amber")], MAX_T_CAL) + pick("false_yes", 1, [("relation", "amber")], MAX_T_CAL)
    + pick("false_yes", 2, [("relation", "reefknot")], MAX_T_CAL)
    + pick("true_yes", 1, [("attribute", "amber")], MAX_T_CAL) + pick("true_no", 1, [("relation", "reefknot")], MAX_T_CAL)
)
assert len(cal_set) >= 6, "too few calibration questions under MAX_T_CAL"
seed_set = cal_set[:N_SEED_Q]
all_check = list(dict.fromkeys(acc_set + cal_set))
print(len(acc_set), "accuracy questions,", len(cal_set), "calibration questions")

dla_ld = {}
dla_paths = sorted(glob.glob(f"{INPUT}/dla-intern/*_batch_*.pt"))
need_ids = set(all_check)
for i, path in enumerate(dla_paths):
    for e in torch.load(path):
        if e["question_id"] in need_ids:
            dla_ld[e["question_id"]] = float(e["logit_diff"])
    print(f"dla-intern batch {i + 1}/{len(dla_paths)}: {len(dla_ld)}/{len(need_ids)} found", flush=True)
    if len(dla_ld) == len(need_ids):
        break
assert set(dla_ld) == need_ids, "dla-intern is missing some checked questions"

prepped = {qid: prep(qid) for qid in all_check}
print("prepared", len(prepped), "questions", flush=True)
p0 = prepped[cal_set[0]]
print("prompt tail:", processor.tokenizer.convert_ids_to_tokens(p0["inputs"]["input_ids"][0, -TAIL_N - 3:].tolist()))
print("positions:", {g: int(p0["masks"][g].sum()) for g in GROUPS}, "of", p0["T"])
print("sequence length: accuracy questions", min(prepped[q]["T"] for q in acc_set), "to", max(prepped[q]["T"] for q in acc_set),
      "| calibration questions", min(prepped[q]["T"] for q in cal_set), "to", max(prepped[q]["T"] for q in cal_set))

108249 Visual Genome images indexed
amber_ [9454] ['Yes'] [2753] ['No']
reefknot_ [9693, 9454] ['yes', 'Yes'] [2753, 2152] ['No', 'no']
4292 selected questions
1957 reefknot images needed, 0 missing
36 accuracy questions, 8 calibration questions
dla-intern batch 1/36: 2/44 found
dla-intern batch 2/36: 3/44 found
dla-intern batch 3/36: 3/44 found
dla-intern batch 4/36: 3/44 found
dla-intern batch 5/36: 4/44 found
dla-intern batch 6/36: 5/44 found
dla-intern batch 7/36: 7/44 found
dla-intern batch 8/36: 8/44 found
dla-intern batch 9/36: 8/44 found
dla-intern batch 10/36: 8/44 found
dla-intern batch 11/36: 9/44 found
dla-intern batch 12/36: 9/44 found
dla-intern batch 13/36: 12/44 found
dla-intern batch 14/36: 12/44 found
dla-intern batch 15/36: 12/44 found
dla-intern batch 16/36: 14/44 found
dla-intern batch 17/36: 16/44 found
dla-intern batch 18/36: 17/44 found
dla-intern batch 19/36: 17/44 found
dla-intern batch 20/36: 17/44 found
dla-intern batch 21/36: 19/44 found
dla-intern batch 22

In [4]:
STATE = {"noise": None, "seed": 0, "capture": False, "grad": False, "live": None, "vis": None, "patch": None, "final_h": None, "probe": None}


def _layer_pre(l):
    def hook(module, args, kwargs):
        h = args[0] if args else kwargs["hidden_states"]
        changed = False
        if l == 0:
            if STATE["noise"] is not None:
                vis = STATE["vis"]
                h = h.clone()
                x = h[:, vis].float()
                generator = torch.Generator().manual_seed(STATE["seed"])
                noise = torch.randn(x.shape, generator=generator).to(h.device)
                noisy = x + noise * x.std() * STATE["noise"]
                h[:, vis] = noisy.to(h.dtype)
                changed = True
                if STATE["probe"] is not None:
                    STATE["probe"]["embed"] = (float(x.std()), float(x.abs().max()), float(noisy.abs().max()), float(h.float().abs().max()))
            if STATE["grad"]:
                h = h.detach().requires_grad_(True)
                changed = True
        if STATE["probe"] is not None:
            STATE["probe"].setdefault("layers", []).append(float(h.float().abs().max()))
        if STATE["capture"]:
            STATE["live"]["rp"][l] = h
        if not changed:
            return None
        if args:
            return (h,) + tuple(args[1:]), kwargs
        kwargs = dict(kwargs)
        kwargs["hidden_states"] = h
        return args, kwargs
    return hook


def _z_pre(l):
    def hook(module, args):
        z = args[0]
        patch = STATE["patch"]
        if patch is not None and patch["kind"] == "head" and patch["layer"] == l:
            T = z.shape[1]
            z4 = z.clone().view(1, T, n_head, head_dim)
            src = patch["src"].view(T, n_head, head_dim)
            idx = patch["idx"].to(z.device)
            z4[:, idx, patch["head"], :] = src[idx.to(src.device), patch["head"], :].to(z.device, z.dtype)
            z = z4.view(1, T, n_head * head_dim)
        if STATE["capture"]:
            STATE["live"]["z"][l] = z
        return (z,) + tuple(args[1:])
    return hook


def _mlp_fwd(l):
    def hook(module, args, output):
        out = output
        patch = STATE["patch"]
        if patch is not None and patch["kind"] == "mlp" and patch["layer"] == l:
            out = out.clone()
            idx = patch["idx"].to(out.device)
            out[:, idx] = patch["src"][idx.to(patch["src"].device)].to(out.device, out.dtype).unsqueeze(0)
        if STATE["capture"]:
            STATE["live"]["mlp"][l] = out
        return out
    return hook


def _final_norm_pre(module, args):
    if STATE["capture"]:
        STATE["final_h"] = args[0][0, -1].detach().float()


_handles = []
for l, layer in enumerate(layers):
    _handles.append(layer.register_forward_pre_hook(_layer_pre(l), with_kwargs=True))
    _handles.append(layer.self_attn.o_proj.register_forward_pre_hook(_z_pre(l)))
    _handles.append(layer.mlp.register_forward_hook(_mlp_fwd(l)))
_handles.append(ln_f.register_forward_pre_hook(_final_norm_pre))
print(len(_handles), "hooks registered")

_KW = {}


def _save_kw(module, args, kwargs):
    _KW["args"], _KW["kwargs"] = args, kwargs


layers[0].register_forward_pre_hook(_save_kw, with_kwargs=True)

_FH = {}


def _save_final(module, args):
    _FH["h"] = args[0][0, -1].detach()


ln_f.register_forward_pre_hook(_save_final)


85 hooks registered


In [5]:
def mask_on(p, gname, dev):
    cache = p.setdefault("_mask", {})
    key = (gname, str(dev))
    if key not in cache:
        cache[key] = p["masks"][gname].to(dev)
    return cache[key]


def group_idx(p, gname):
    if gname == "all":
        return torch.arange(p["T"])
    return torch.nonzero(p["masks"][gname] > 0)[:, 0]


def make_patch(kind, layer, idx, src, head=None):
    return {"kind": kind, "layer": layer, "idx": idx, "src": src, "head": head}


def read_delta(p, out):
    logits = out.logits[0, -1].float()
    return p["sign"] * (logits[p["yes_ids"]].mean() - logits[p["no_ids"]].mean())


def capture_run_full(p, noise, seed, backward):
    STATE.update(noise=noise, seed=seed, capture=True, grad=backward, vis=p["vis"], patch=None, final_h=None)
    STATE["live"] = {"rp": [None] * n_layer, "z": [None] * n_layer, "mlp": [None] * n_layer}
    grads, finite = None, True
    with torch.set_grad_enabled(backward):
        out = model(**p["inputs"], use_cache=False, logits_to_keep=1)
        delta = read_delta(p, out)
        live = STATE["live"]
        if backward:
            raw = torch.autograd.grad(delta * GRAD_SCALE, live["z"] + live["mlp"] + live["rp"])
            finite = all(bool(torch.isfinite(g).all()) for g in raw)
            grads = {"z": list(raw[:n_layer]), "mlp": list(raw[n_layer:2 * n_layer]), "rp": list(raw[2 * n_layer:])}
    acts = {k: [t.detach() for t in live[k]] for k in ("z", "mlp", "rp")}
    result = {"acts": acts, "grads": grads, "delta": float(delta.detach()), "final_h": STATE["final_h"], "finite": finite}
    STATE.update(capture=False, grad=False, live=None, noise=None, final_h=None)
    del out, delta, live
    return result


def capture_run(p, noise, seed, backward=True):
    """Memory-light version: forward without a graph, then per-layer recompute for the backward pass."""
    STATE.update(noise=noise, seed=seed, capture=True, grad=False, vis=p["vis"], patch=None, final_h=None)
    STATE["live"] = {"rp": [None] * n_layer, "z": [None] * n_layer, "mlp": [None] * n_layer}
    with torch.no_grad():
        out = model(**p["inputs"], use_cache=False, logits_to_keep=1)
        delta = float(read_delta(p, out))
    acts = {k: [t.detach() for t in STATE["live"][k]] for k in ("z", "mlp", "rp")}
    kw_args, kw_kwargs, h_last = _KW["args"], _KW["kwargs"], _FH["h"]
    final_h = h_last.float()
    del out
    if not backward:
        STATE.update(capture=False, live=None, noise=None, final_h=None)
        return {"acts": acts, "grads": None, "delta": delta, "final_h": final_h, "finite": True}
    STATE.update(capture=False, live=None, noise=None)
    with torch.enable_grad():
        hl = h_last.clone().requires_grad_(True)
        logits = lm_head(ln_f(hl.view(1, 1, -1)))[0, -1].float()
        d = p["sign"] * (logits[p["yes_ids"]].mean() - logits[p["no_ids"]].mean())
        g_last = torch.autograd.grad(d * GRAD_SCALE, hl)[0]
    finite = bool(torch.isfinite(g_last).all())
    gz, gm, gr = [None] * n_layer, [None] * n_layer, [None] * n_layer
    g_carry = None
    for l in range(n_layer - 1, -1, -1):
        STATE.update(capture=True, grad=False, noise=None, patch=None)
        STATE["live"] = {"rp": [None] * n_layer, "z": [None] * n_layer, "mlp": [None] * n_layer}
        with torch.enable_grad():
            h_in = acts["rp"][l].detach().clone().requires_grad_(True)
            if kw_args:
                o = layers[l](h_in, *kw_args[1:], **kw_kwargs)
            else:
                kw2 = dict(kw_kwargs)
                kw2["hidden_states"] = h_in
                o = layers[l](**kw2)
            o = o[0] if isinstance(o, tuple) else o
            z_l, m_l = STATE["live"]["z"][l], STATE["live"]["mlp"][l]
            if g_carry is None:
                go = torch.zeros_like(o)
                go[0, -1] = g_last.to(o.device, o.dtype)
            else:
                go = g_carry.to(o.device, o.dtype)
            gz_, gm_, gh_ = torch.autograd.grad(o, [z_l, m_l, h_in], grad_outputs=go)
        gz[l], gm[l], gr[l], g_carry = gz_, gm_, gh_, gh_
        finite = finite and all(bool(torch.isfinite(t).all()) for t in (gz_, gm_, gh_))
        STATE["live"] = None
        del o, z_l, m_l, h_in, go
    STATE.update(capture=False, grad=False, live=None, noise=None)
    return {"acts": acts, "grads": {"z": gz, "mlp": gm, "rp": gr}, "delta": delta, "final_h": final_h, "finite": finite}


@torch.no_grad()
def delta_with(p, noise, seed, patch=None):
    STATE.update(noise=noise, seed=seed, capture=False, grad=False, vis=p["vis"], patch=patch)
    out = model(**p["inputs"], use_cache=False, logits_to_keep=1)
    value = float(read_delta(p, out))
    STATE.update(noise=None, patch=None)
    return value


def atp_effects(A, B, G, p):
    head = torch.zeros(len(GROUPS), n_layer, n_head)
    mlp = torch.zeros(len(GROUPS), n_layer)
    rp = torch.zeros(len(GROUPS), n_layer)
    T = p["T"]
    for l in range(n_layer):
        dz = ((A["z"][l].float() - B["z"][l].float()) * G["z"][l].float() / GRAD_SCALE)[0].view(T, n_head, head_dim).sum(-1)
        dm = ((A["mlp"][l].float() - B["mlp"][l].float()) * G["mlp"][l].float() / GRAD_SCALE)[0].sum(-1)
        dr = ((A["rp"][l].float() - B["rp"][l].float()) * G["rp"][l].float() / GRAD_SCALE)[0].sum(-1)
        for gi, g in enumerate(GROUPS):
            head[gi, l] = (mask_on(p, g, dz.device) @ dz).cpu()
            mlp[gi, l] = (mask_on(p, g, dm.device) @ dm).cpu()
            rp[gi, l] = (mask_on(p, g, dr.device) @ dr).cpu()
    return {"head": head, "mlp": mlp, "rp": rp}

In [10]:
import sys, gc
sys.last_traceback = None
sys.last_value = None
sys.last_type = None
for _v in ("a", "b", "c", "c_", "r"):
    globals().pop(_v, None)
STATE.update(noise=None, capture=False, grad=False, live=None, patch=None, final_h=None, probe=None)
gc.collect()
torch.cuda.empty_cache()
for i in range(torch.cuda.device_count()):
    print(f"GPU{i}: allocated {torch.cuda.memory_allocated(i) / 1e9:.1f} GB, reserved {torch.cuda.memory_reserved(i) / 1e9:.1f} GB")

GPU0: allocated 8.1 GB, reserved 8.6 GB
GPU1: allocated 7.9 GB, reserved 8.0 GB


In [7]:
import gc
pp = {}
for p_ in model.parameters():
    pp[str(p_.device)] = pp.get(str(p_.device), 0) + p_.numel() * p_.element_size()
print("weights per device (GB):", {k: round(v / 1e9, 1) for k, v in pp.items()})
for i in range(torch.cuda.device_count()):
    print(f"GPU{i}: allocated {torch.cuda.memory_allocated(i) / 1e9:.1f} GB")

weights per device (GB): {'cuda:0': 8.0, 'cuda:1': 7.9}
GPU0: allocated 13.0 GB
GPU1: allocated 7.9 GB


In [8]:
import gc
n_before = len(_prep_cache)
for k in [k for k in _prep_cache if k not in prepped]:
    del _prep_cache[k]
gc.collect()
torch.cuda.empty_cache()
print(f"prep cache {n_before} -> {len(_prep_cache)}")
for i in range(torch.cuda.device_count()):
    print(f"GPU{i}: allocated {torch.cuda.memory_allocated(i) / 1e9:.1f} GB")

prep cache 457 -> 44
GPU0: allocated 8.1 GB
GPU1: allocated 7.9 GB


In [9]:
import gc
gc.collect()
torch.cuda.empty_cache()
p = prepped[min(all_check, key=lambda q: prepped[q]["T"])]
print("gradient check on", p["qid"], "with", p["T"], "tokens", flush=True)
for scale in GRAD_SCALE_CANDIDATES:
    GRAD_SCALE = scale
    a = capture_run_full(p, None, 0, True)
    if a["finite"]:
        break
    print(f"full backward is not finite at gradient scale {scale}, trying a smaller one", flush=True)
    del a
    torch.cuda.empty_cache()
assert a["finite"], "gradients are not finite at any scale"
b = capture_run(p, None, 0, True)
assert b["finite"], f"checkpointed gradients are not finite at scale {GRAD_SCALE}"
err = 0.0
for k in ("z", "mlp", "rp"):
    for l in range(n_layer):
        ga, gb = a["grads"][k][l].float().cpu(), b["grads"][k][l].float().cpu()
        e = float((ga - gb).abs().max() / ga.abs().max().clamp_min(1e-12))
        assert np.isfinite(e), f"non-finite gradient comparison at {k} layer {l}, gradient scale {GRAD_SCALE}"
        err = max(err, e)
print(f"gradient scale {GRAD_SCALE}: checkpointed vs full backward max relative gradient difference {err:.2e} | delta {a['delta']:+.4f} vs {b['delta']:+.4f}")
assert err < 0.05, "checkpointed gradients disagree with the full backward pass, do not continue"
del a, b
torch.cuda.empty_cache()


gradient check on amber_1189 with 275 tokens


/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can happen if the autograd graph is still being kept alive by tensors such as the loss, or if you are using DDP, which will stash a reference to the node. To resolve the mismatch, delete all references to the autograd graph or ensure that DDP initialization is performed under the same stream as subsequent forwards. If the mismatch is intentional, you can use torch.autograd.graph.set_warn_on_accumulate_grad_stream_mismatch(False) to suppress this warning. (Triggered internally at /pytorch/torch/csrc/autograd/input_buffer.cpp:240.)
  return Variable._execution_en

gradient scale 32.0: checkpointed vs full backward max relative gradient difference 0.00e+00 | delta +5.4062 vs +5.4062


In [11]:
t0 = time.time()
clean_delta = {qid: delta_with(prepped[qid], None, 0) for qid in all_check}
print(f"{len(all_check)} clean forwards in {time.time() - t0:.1f} s ({(time.time() - t0) / len(all_check):.2f} s each)")

err = {qid: abs(clean_delta[qid] * prepped[qid]["sign"] - dla_ld[qid]) for qid in all_check}
print(f"clean logit difference vs dla-intern: max err {max(err.values()):.4f}, mean err {np.mean(list(err.values())):.4f}")
assert all(np.isfinite(list(clean_delta.values()))), "non-finite clean deltas"
assert max(err.values()) < 0.25, "prompt or readout differs from the DLA collection"

acc_combo = [(question_by_id[q]["hallucination_type"], selection["source"][q]) for q in acc_set]
acc_sign = np.array([prepped[q]["sign"] for q in acc_set])
clean_ok = np.array([clean_delta[q] > 0 for q in acc_set])
print(f"clean: accuracy {clean_ok.mean():.2f} on {len(acc_set)} balanced true_yes and true_no questions")
assert clean_ok.mean() > 0.85, "clean accuracy is too low"

noise_acc, noise_finite, noise_gap, noise_lean = {}, {}, {}, {}
for mult in NOISE_CANDIDATES:
    deltas = np.array([delta_with(prepped[qid], mult, 0) for qid in acc_set])
    noise_finite[mult] = bool(np.isfinite(deltas).all())
    if not noise_finite[mult]:
        print(f"noise x{mult}: non-finite runs {int((~np.isfinite(deltas)).sum())}/{len(deltas)}")
        continue
    raw = acc_sign * deltas
    noise_acc[mult] = float(np.mean(deltas > 0))
    noise_gap[mult] = float(np.mean(deltas))
    noise_lean[mult] = float(np.mean(raw))
    print(f"noise x{mult}: accuracy {noise_acc[mult]:.2f}, mean delta {noise_gap[mult]:+.2f} | blind Yes minus No: mean {noise_lean[mult]:+.2f} "
          f"(true_yes {raw[acc_sign > 0].mean():+.2f}, true_no {raw[acc_sign < 0].mean():+.2f}), share answering Yes {np.mean(raw > 0):.2f}")


def last_layer_max(mult):
    vals = []
    for qid in acc_set:
        STATE["probe"] = {}
        delta_with(prepped[qid], mult, 0)
        vals.append(STATE["probe"]["layers"][-1])
        STATE["probe"] = None
    return float(np.max(vals))


usable = []
for m in FINALISTS:
    assert noise_finite.get(m), f"x{m} is not finite"
    peak = last_layer_max(m)
    print(f"noise x{m}: max |residual| entering the last layer over {len(acc_set)} questions {peak:.0f} (fp16 limit 65504, required below {HEADROOM_MAX:.0f})")
    if peak < HEADROOM_MAX:
        usable.append(m)
assert usable, "no finalist has fp16 headroom"
print("finalists carried to calibration:", usable)

p = prepped[cal_set[0]]
for scale in GRAD_SCALE_CANDIDATES:
    GRAD_SCALE = scale
    c = capture_run(p, None, 0, True)
    fin = c["finite"]
    del c
    torch.cuda.empty_cache()
    print(f"gradient scale {scale}: all gradients finite {fin}")
    if fin:
        break
assert fin, "gradients are not finite at any scale"
GRAD_SCALE_USED = GRAD_SCALE

44 clean forwards in 30.4 s (0.69 s each)
clean logit difference vs dla-intern: max err 0.0156, mean err 0.0011
clean: accuracy 1.00 on 36 balanced true_yes and true_no questions
noise x3.0: accuracy 0.58, mean delta +0.20 | blind Yes minus No: mean -0.30 (true_yes -0.10, true_no -0.50), share answering Yes 0.42
noise x4.0: accuracy 0.50, mean delta +0.09 | blind Yes minus No: mean -0.87 (true_yes -0.78, true_no -0.96), share answering Yes 0.28
noise x5.0: accuracy 0.50, mean delta +0.02 | blind Yes minus No: mean -1.23 (true_yes -1.21, true_no -1.25), share answering Yes 0.17
noise x6.0: accuracy 0.47, mean delta -0.02 | blind Yes minus No: mean -1.51 (true_yes -1.54, true_no -1.49), share answering Yes 0.14
noise x3.0: max |residual| entering the last layer over 36 questions 4776 (fp16 limit 65504, required below 20000)
noise x4.0: max |residual| entering the last layer over 36 questions 4776 (fp16 limit 65504, required below 20000)
noise x6.0: max |residual| entering the last layer 

In [12]:
MEM_LIMIT_GB = 14.0
MEM_PROBE = []
MAX_FIT = None
GRAD_SCALE = GRAD_SCALE_USED
by_tokens = {}
for qid in all_sel:
    n_tok = selection["n_image_tokens"][qid]
    by_tokens.setdefault(n_tok, []).append(qid)
print("selected questions by image token count:", {k: len(v) for k, v in sorted(by_tokens.items())})
for n_tok in sorted(by_tokens):
    qid = next((q for q in by_tokens[n_tok] if (lambda pth: pth is not None and os.path.exists(pth))(resolve_image_path(question_by_id[q]))), None)
    if qid is None:
        continue
    p = prep(qid)
    for i in range(torch.cuda.device_count()):
        torch.cuda.reset_peak_memory_stats(i)
    t_probe = time.time()
    try:
        c_ = capture_run(p, None, 0, True)
        ok = bool(c_["finite"])
        del c_
    except torch.cuda.OutOfMemoryError:
        STATE.update(capture=False, grad=False, live=None, noise=None, final_h=None)
        ok = False
    peaks = [torch.cuda.max_memory_allocated(i) / 1e9 for i in range(torch.cuda.device_count())]
    torch.cuda.empty_cache()
    MEM_PROBE.append({"n_image_tokens": n_tok, "T": p["T"], "peak_gb": peaks, "ok": ok})
    print(f"{n_tok} image tokens, T {p['T']}: grad run ok {ok} in {time.time() - t_probe:.1f} s, peak memory per GPU {[round(x, 1) for x in peaks]} GB", flush=True)
    if not ok or max(peaks, default=0) > MEM_LIMIT_GB:
        break
    MAX_FIT = n_tok
print(f"largest image token count that fits under {MEM_LIMIT_GB} GB per GPU: {MAX_FIT}")
if MAX_FIT is not None:
    n_over = sum(selection["n_image_tokens"][q] > MAX_FIT for q in all_sel)
    print(f"{n_over} of {len(all_sel)} selected questions exceed it; set MAX_IMAGE_TOKENS = {MAX_FIT} in the selection notebook and re-run it before collection if that share is acceptable")
for p_ in prepped.values():
    p_.pop("_mask", None)

selected questions by image token count: {256: 212, 768: 196, 1024: 6, 1280: 27, 1792: 1946, 2304: 18, 2560: 11, 2816: 14, 3328: 1862}
256 image tokens, T 276: grad run ok True in 1.0 s, peak memory per GPU [8.3, 8.2] GB
768 image tokens, T 787: grad run ok True in 2.3 s, peak memory per GPU [8.8, 8.8] GB
1024 image tokens, T 1043: grad run ok True in 3.5 s, peak memory per GPU [9.1, 9.2] GB
1280 image tokens, T 1301: grad run ok True in 4.3 s, peak memory per GPU [9.5, 9.6] GB
1792 image tokens, T 1812: grad run ok True in 6.6 s, peak memory per GPU [10.5, 10.7] GB
2304 image tokens, T 2323: grad run ok True in 9.5 s, peak memory per GPU [11.7, 12.0] GB
2560 image tokens, T 2580: grad run ok True in 11.3 s, peak memory per GPU [12.4, 12.7] GB
2816 image tokens, T 2843: grad run ok True in 13.1 s, peak memory per GPU [13.2, 13.5] GB
3328 image tokens, T 3347: grad run ok False in 10.9 s, peak memory per GPU [12.1, 15.1] GB
largest image token count that fits under 14.0 GB per GPU: 2816

In [13]:
SEED0 = 1000
CAL = {}
TIMING = {"capture": [], "exact": []}
u_cache = {}


def u_for(p):
    key = (tuple(p["yes_ids"]), tuple(p["no_ids"]))
    if key not in u_cache:
        dev = lm_head.weight.device
        yes = lm_head.weight[p["yes_ids"]].float().mean(0)
        no = lm_head.weight[p["no_ids"]].float().mean(0)
        u_cache[key] = (ln_f.weight.float().to(dev) * (yes - no)).cpu()
    return u_cache[key]


def calibrate(level):
    results = []
    for qi, qid in enumerate(cal_set):
        p = prepped[qid]
        seed = SEED0 + qi
        print(f"x{level} [{qi + 1}/{len(cal_set)}] {qid} T={p['T']}: grad run + noised run ...", flush=True)
        t0 = time.time()
        c = capture_run(p, None, seed, True)
        r = capture_run(p, level, seed, False)
        TIMING["capture"].append(time.time() - t0)
        assert c["finite"], f"non-finite gradients for {qid}: lower the gradient scale"
        assert np.isfinite(r["delta"]) and np.isfinite(c["delta"]), f"non-finite delta for {qid}"
        TE = c["delta"] - r["delta"]
        atp = atp_effects(r["acts"], c["acts"], c["grads"], p)
        selfcheck = abs(delta_with(p, level, seed) - r["delta"])

        h = c["final_h"]
        rms = torch.sqrt(h.pow(2).mean() + ln_f.variance_epsilon)
        dm = (r["acts"]["mlp"][-1][0, -1].float() - c["acts"]["mlp"][-1][0, -1].float()).to(h.device)
        anchor = float(p["sign"] * (dm @ u_for(p).to(h.device)) / rms)

        head_all = atp["head"].sum(0).flatten()
        order = torch.argsort(head_all.abs(), descending=True).tolist()
        top = order[:N_HEAD_TOP]
        near = order[N_HEAD_TOP:N_HEAD_TOP + N_HEAD_RAND]
        exact = {"mlp": {}, "head": {}}
        t1 = time.time()
        n_fwd = 0
        n_total = n_layer * (1 + len(GROUPS)) + N_HEAD_TOP + N_HEAD_RAND

        def _progress():
            if n_fwd % 15 == 0 or n_fwd == n_total:
                per = (time.time() - t1) / n_fwd
                print(f"    x{level} q{qi + 1}/{len(cal_set)} exact patching {n_fwd}/{n_total} "
                      f"({per:.2f} s/fwd, ~{per * (n_total - n_fwd) / 60:.1f} min left on this question)", flush=True)

        for l in range(n_layer):
            for gname in ["all"] + GROUPS:
                patch = make_patch("mlp", l, group_idx(p, gname), r["acts"]["mlp"][l][0])
                exact["mlp"][(gname, l)] = delta_with(p, None, seed, patch) - c["delta"]
                n_fwd += 1
                _progress()
        for flat in top + near:
            l, h_ = divmod(int(flat), n_head)
            patch = make_patch("head", l, group_idx(p, "all"), r["acts"]["z"][l][0], h_)
            exact["head"][(l, h_)] = delta_with(p, None, seed, patch) - c["delta"]
            n_fwd += 1
            _progress()
        TIMING["exact"].append((time.time() - t1) / n_fwd)

        results.append({
            "qid": qid, "stratum": selection["stratum"][qid], "TE": TE, "delta_clean": c["delta"], "delta_corrupt": r["delta"],
            "blind_gap": p["sign"] * r["delta"], "selfcheck": selfcheck, "anchor": anchor, "atp": atp, "exact": exact, "near": set(near), "T": p["T"],
        })
        print(f"x{level} [{qi + 1}/{len(cal_set)}] {qid} {selection['stratum'][qid]} T={p['T']}: clean {c['delta']:+.2f} corrupt {r['delta']:+.2f} TE {TE:+.2f} | "
              f"grad run {TIMING['capture'][-1]:.1f} s, exact {TIMING['exact'][-1]:.2f} s/forward", flush=True)
        del c, r, atp
        torch.cuda.empty_cache()
    return results


for level in usable:
    print(f"=== calibrating noise x{level} ({usable.index(level) + 1}/{len(usable)}) ===", flush=True)
    CAL[level] = calibrate(level)


=== calibrating noise x3.0 (1/3) ===
x3.0 [1/8] amber_6748 T=276: grad run + noised run ...
    x3.0 q1/8 exact patching 15/152 (0.32 s/fwd, ~0.7 min left on this question)
    x3.0 q1/8 exact patching 30/152 (0.32 s/fwd, ~0.7 min left on this question)
    x3.0 q1/8 exact patching 45/152 (0.32 s/fwd, ~0.6 min left on this question)
    x3.0 q1/8 exact patching 60/152 (0.32 s/fwd, ~0.5 min left on this question)
    x3.0 q1/8 exact patching 75/152 (0.33 s/fwd, ~0.4 min left on this question)
    x3.0 q1/8 exact patching 90/152 (0.33 s/fwd, ~0.3 min left on this question)
    x3.0 q1/8 exact patching 105/152 (0.33 s/fwd, ~0.3 min left on this question)
    x3.0 q1/8 exact patching 120/152 (0.33 s/fwd, ~0.2 min left on this question)
    x3.0 q1/8 exact patching 135/152 (0.33 s/fwd, ~0.1 min left on this question)
    x3.0 q1/8 exact patching 150/152 (0.33 s/fwd, ~0.0 min left on this question)
    x3.0 q1/8 exact patching 152/152 (0.33 s/fwd, ~0.0 min left on this question)
x3.0 [1/8] a

In [14]:
def pair_frame(cal_results):
    rows = []
    for res in cal_results:
        a, e = res["atp"], res["exact"]
        mlp_all, head_all = a["mlp"].sum(0), a["head"].sum(0)
        for (g, l), v in e["mlp"].items():
            atp_v = mlp_all[l].item() if g == "all" else a["mlp"][GROUPS.index(g), l].item()
            rows.append({"qid": res["qid"], "kind": f"mlp_{g}", "comp": l, "atp": atp_v, "exact": v, "pool": "mlp"})
        for (l, h_), v in e["head"].items():
            rows.append({"qid": res["qid"], "kind": "head_all", "comp": l * n_head + h_, "atp": head_all[l, h_].item(),
                         "exact": v, "pool": "near" if l * n_head + h_ in res["near"] else "top"})
    return pd.DataFrame(rows)


def calib_stats(df):
    no_text = df[df.kind != "mlp_text"]
    out = {
        "pooled_spearman": float(spearmanr(df.atp, df.exact)[0]),
        "pooled_spearman_no_text": float(spearmanr(no_text.atp, no_text.exact)[0]),
        "slope_exact_on_atp": float((df.atp * df.exact).sum() / (df.atp ** 2).sum()),
    }
    for kind, sub in df.groupby("kind"):
        out[f"spearman_{kind}"] = float(spearmanr(sub.atp, sub.exact)[0])
    key = df[df.kind.isin(["mlp_all", "head_all"])]
    rec10, recK, sgn, perq, miss = [], [], [], [], []
    for qid, sub in key.groupby("qid"):
        sub = sub.reset_index(drop=True)
        k1, k2 = min(10, len(sub)), min(RECALL_K, len(sub))
        rec10.append(len(set(sub.exact.abs().nlargest(k1).index) & set(sub.atp.abs().nlargest(k1).index)) / k1)
        recK.append(len(set(sub.exact.abs().nlargest(k2).index) & set(sub.atp.abs().nlargest(k2).index)) / k2)
        big = sub[sub.exact.abs() > 0.1 * sub.exact.abs().max()]
        sgn.append(float((np.sign(big.atp) == np.sign(big.exact)).mean()))
        perq.append(float(spearmanr(sub.atp, sub.exact)[0]))
        heads = sub[sub.kind == "head_all"]
        th, nh = heads[heads.pool == "top"], heads[heads.pool == "near"]
        if len(th) and len(nh):
            thr = th.exact.abs().nlargest(min(10, len(th))).iloc[-1]
            miss.append(float((nh.exact.abs() > thr).sum()))
    out.update(top10_recall=float(np.mean(rec10)), **{f"top{RECALL_K}_recall": float(np.mean(recK))},
               sign_agreement=float(np.mean(sgn)), per_question_spearman=float(np.mean(perq)),
               near_miss_heads=float(np.mean(miss)) if miss else float("nan"))
    return out


def mlp_sign_agreement(df):
    big = df[(df.kind == "mlp_all") & (df.exact.abs() > 0.5)]
    return float((np.sign(big.atp) == np.sign(big.exact)).mean()) if len(big) else float("nan")


layers_rp = sorted({0, n_layer // 4, n_layer // 2, 3 * n_layer // 4, n_layer - 1})
calib, table = {}, []
for level, res in CAL.items():
    df_l = pair_frame(res)
    s = calib_stats(df_l)
    s["mlp_sign_agreement_big"] = mlp_sign_agreement(df_l)
    s["blind_lean"] = float(np.mean([r_["blind_gap"] for r_ in res]))
    s["median_abs_TE"] = float(np.median([abs(r_["TE"]) for r_ in res]))
    s["frac_TE_below_minus1_false_yes"] = float(np.mean([r_["TE"] < -1 for r_ in res if r_["stratum"] == "false_yes"]))
    use = [r_ for r_ in res if abs(r_["TE"]) > MIN_TE]
    for l in layers_rp:
        s[f"rp_ratio_L{l}"] = float(np.median([r_["atp"]["rp"].sum(0)[l].item() / (-r_["TE"]) for r_ in use])) if use else float("nan")
    calib[level] = s
    table.append({"noise": level, "lean": s["blind_lean"], "acc": noise_acc[level], "pooled_no_text": s["pooled_spearman_no_text"],
                  "mlp_all": s["spearman_mlp_all"], "mlp_visual": s["spearman_mlp_visual"], "head": s["spearman_head_all"],
                  f"top{RECALL_K}": s[f"top{RECALL_K}_recall"], "sign": s["sign_agreement"], "mlp_sign_big": s["mlp_sign_agreement_big"],
                  "near_miss": s["near_miss_heads"], "slope": s["slope_exact_on_atp"], "median|TE|": s["median_abs_TE"]})
print(pd.DataFrame(table).round(3).to_string(index=False))
print(f"blind lean = mean blind-run Yes minus No over the {len(cal_set)} calibration questions (0 means the blind model has no preference)")

eligible = [m for m in CAL if abs(calib[m]["blind_lean"]) <= LEAN_MAX]
pool_ = eligible if eligible else list(CAL)
NOISE_MULT = max(pool_, key=lambda m: calib[m]["pooled_spearman_no_text"]) if eligible else min(pool_, key=lambda m: abs(calib[m]["blind_lean"]))
print(f"levels with |blind lean| <= {LEAN_MAX}: {eligible}; using NOISE_MULT = {NOISE_MULT}")
s = calib[NOISE_MULT]
meets = bool(s["pooled_spearman_no_text"] >= 0.8 and s[f"top{RECALL_K}_recall"] >= 0.7 and s["near_miss_heads"] <= 2)
print(f"chosen level meets the bars (pooled spearman without text >= 0.8, top-{RECALL_K} recall >= 0.7, near-miss heads <= 2): {meets}")

cal_results = CAL[NOISE_MULT]
mid = min(n_layer // 2 + 2, n_layer - 1)
print(f"--- x{NOISE_MULT}: L{mid} MLP, all positions: atp vs exact per question ---")
for res in cal_results:
    print(f"{res['qid']:24s} {res['stratum']:10s} TE {res['TE']:+7.2f} blind gap {res['blind_gap']:+6.2f} | atp {res['atp']['mlp'].sum(0)[mid]:+7.3f} exact {res['exact']['mlp'][('all', mid)]:+7.3f}")
print(f"--- x{NOISE_MULT}: residual stream identity (atp of resid_pre at all positions / -TE; exact is 1) ---")
print({l: round(s[f"rp_ratio_L{l}"], 2) for l in layers_rp})
print(f"--- x{NOISE_MULT}: anchor, last-layer MLP at all positions ---")
for res in cal_results:
    print(f"{res['qid']:24s} atp {res['atp']['mlp'].sum(0)[-1]:+7.3f} dla {res['anchor']:+7.3f} exact {res['exact']['mlp'][('all', n_layer - 1)]:+7.3f}")
print(f"grad run vs no-grad run delta, max abs diff: {max(r_['selfcheck'] for lv in CAL.values() for r_ in lv):.4f}")

 noise   lean   acc  pooled_no_text  mlp_all  mlp_visual  head  top20  sign  mlp_sign_big  near_miss  slope  median|TE|
   3.0  0.089 0.583           0.748    0.631       0.435 0.965  0.688 0.918         0.833      0.375  0.651       3.037
   4.0 -0.252 0.500           0.714    0.571       0.335 0.959  0.694 0.905         0.792      1.250  0.511       3.711
   6.0 -0.377 0.472           0.650    0.453       0.264 0.957  0.688 0.900         0.745      0.500  0.350       3.367
blind lean = mean blind-run Yes minus No over the 8 calibration questions (0 means the blind model has no preference)
levels with |blind lean| <= 2.0: [3.0, 4.0, 6.0]; using NOISE_MULT = 3.0
chosen level meets the bars (pooled spearman without text >= 0.8, top-20 recall >= 0.7, near-miss heads <= 2): False
--- x3.0: L16 MLP, all positions: atp vs exact per question ---
amber_6748               false_yes  TE   -2.39 blind gap  +0.22 | atp  -1.634 exact  -2.094
amber_5478               false_yes  TE   -0.48 blind gap

In [15]:
vec = lambda a: torch.cat([a["head"].sum(0).flatten(), a["mlp"].sum(0)]).numpy()

seed_stab, te_spread = [], []
for qid in seed_set:
    p = prepped[qid]
    c = capture_run(p, None, 0, True)
    runs, tes = [], []
    for s_ in range(3):
        r = capture_run(p, NOISE_MULT, 100 + s_, False)
        runs.append(vec(atp_effects(r["acts"], c["acts"], c["grads"], p)))
        tes.append(c["delta"] - r["delta"])
        del r
    seed_stab.append(np.mean([spearmanr(runs[i], runs[j])[0] for i in range(3) for j in range(i + 1, 3)]))
    te_spread.append(float(np.std(tes)))
    del c
    torch.cuda.empty_cache()
print(f"rank correlation of effects across noise seeds, mean {np.mean(seed_stab):.3f} (per question {np.round(seed_stab, 3).tolist()})")
print("std of TE across seeds per question:", np.round(te_spread, 3).tolist())

grad_scale_err = None
if GRAD_SCALE_USED != 1.0:
    p = prepped[seed_set[0]]
    r = capture_run(p, NOISE_MULT, 100, False)
    vecs = {}
    for scale in (GRAD_SCALE_USED, 1.0):
        GRAD_SCALE = scale
        c = capture_run(p, None, 0, True)
        vecs[scale] = vec(atp_effects(r["acts"], c["acts"], c["grads"], p)) if c["finite"] else None
        del c
        torch.cuda.empty_cache()
    GRAD_SCALE = GRAD_SCALE_USED
    if vecs[1.0] is not None and np.isfinite(vecs[1.0]).all():
        ref_ = vecs[GRAD_SCALE_USED]
        grad_scale_err = float(np.max(np.abs(vecs[1.0] - ref_)) / np.max(np.abs(ref_)))
        print(f"grad scale 1 vs {GRAD_SCALE_USED}: max difference as a fraction of the largest effect {grad_scale_err:.4f}")
    else:
        print("grad scale 1 is not finite or underflows; the larger scale is required")

rank correlation of effects across noise seeds, mean 0.922 (per question [0.913, 0.942, 0.91])
std of TE across seeds per question: [0.581, 0.16, 0.591]
grad scale 1 vs 32.0: max difference as a fraction of the largest effect 0.0012


In [16]:
summary = {
    "memory_probe": MEM_PROBE, "max_image_tokens_fit": MAX_FIT,
    "noise_accuracy": {str(k): v for k, v in noise_acc.items()},
    "noise_mean_delta": {str(k): v for k, v in noise_gap.items()},
    "noise_blind_lean": {str(k): v for k, v in noise_lean.items()},
    "noise_mult": NOISE_MULT, "tail_n": TAIL_N, "grad_scale": GRAD_SCALE_USED,
    "clean_vs_dla_max_err": float(max(err.values())),
    "calibration_by_level": {str(k): v for k, v in calib.items()},
    "seed_stability": float(np.mean(seed_stab)),
    "grad_scale_diff_fraction_of_max": grad_scale_err,
    "sec_per_grad_run": float(np.mean(TIMING["capture"])),
    "sec_per_exact_forward": float(np.mean(TIMING["exact"])),
    "max_gpu_mem_gb": [torch.cuda.max_memory_allocated(i) / 1e9 for i in range(torch.cuda.device_count())],
}
for key, val in summary.items():
    print(key, val)

sec = summary["sec_per_grad_run"] * 0.6 + summary["sec_per_exact_forward"]
print(f"collection estimate: about {sec:.1f} s per question, {len(all_sel) * sec / 3600:.1f} h for all {len(all_sel)} selected questions")

with open("/kaggle/working/atp_validate_intern.json", "w") as f:
    json.dump(summary, f, indent=1)
print(f"collection notebook settings: NOISE_LEVELS = [{NOISE_MULT}], GRAD_SCALES = [{GRAD_SCALE_USED}, 1.0]")

memory_probe [{'n_image_tokens': 256, 'T': 276, 'peak_gb': [8.347707392, 8.206759936], 'ok': True}, {'n_image_tokens': 768, 'T': 787, 'peak_gb': [8.795908608, 8.76795648], 'ok': True}, {'n_image_tokens': 1024, 'T': 1043, 'peak_gb': [9.140233216, 9.155193344], 'ok': True}, {'n_image_tokens': 1280, 'T': 1301, 'peak_gb': [9.53626624, 9.606982144], 'ok': True}, {'n_image_tokens': 1792, 'T': 1812, 'peak_gb': [10.51093504, 10.673854976], 'ok': True}, {'n_image_tokens': 2304, 'T': 2323, 'peak_gb': [11.725962752, 11.978046976], 'ok': True}, {'n_image_tokens': 2560, 'T': 2580, 'peak_gb': [12.425278976, 12.725462528], 'ok': True}, {'n_image_tokens': 2816, 'T': 2843, 'peak_gb': [13.209826816, 13.545362944], 'ok': True}, {'n_image_tokens': 3328, 'T': 3347, 'peak_gb': [12.1196032, 15.075508736], 'ok': False}]
max_image_tokens_fit 2816
noise_accuracy {'3.0': 0.5833333333333334, '4.0': 0.5, '5.0': 0.5, '6.0': 0.4722222222222222}
noise_mean_delta {'3.0': 0.2004123263888889, '4.0': 0.08832465277777778,